# Bayesian Model Comparison: SIR vs. SEIR

## Introduction
When dealing with complex physical or biological systems, we often have competing mathematical models that explain the same phenomena. A critical question arises: **which model best explains our observed data without unnecessarily overfitting?**

This case study introduces **Bayesian Model Comparison**. We will evaluate two competing epidemiological models—the standard SIR model and the extended SEIR model—against a single set of observed data. 

To do this efficiently, we will use **AutoEmulate** to train fast Gaussian Process surrogate models for both simulators. We will then perform Bayesian calibration to obtain posterior distributions and, crucially, compute the **Bayesian Evidence (Marginal Likelihood)** to calculate the Bayes Factor, allowing us to quantitatively select the superior model.

### The Models
1. **SIR Model:** A simple epidemic model dividing the population into Susceptible $S(t)$, Infectious $I(t)$, and Recovered $R(t)$. It relies on two parameters:
   * $\beta$: Transmission rate per day
   * $\gamma$: Recovery rate per day
2. **SEIR Model:** An extension that adds an Exposed $E(t)$ compartment for individuals who are infected but not yet infectious. It adds a third parameter:
   * $\sigma$: Incubation rate per day

### The Goal
We will generate synthetic "ground truth" observations using the more complex **SEIR model**. We will then calibrate both the SIR and SEIR emulators against this data. Finally, we will use the computed evidence to see if the Bayes Factor correctly penalizes the simpler SIR model for failing to account for the incubation period.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pyro
import pyro.distributions as dist

from pyro.infer import MCMC
from pyro.infer.mcmc import RandomWalkKernel

from autoemulate.simulations.epidemic import Epidemic
from autoemulate.simulations.seir import SEIRSimulator
from autoemulate.core.compare import AutoEmulate
from autoemulate.emulators import GaussianProcessRBF
from autoemulate.calibration.bayes import BayesianCalibration
from autoemulate.calibration.evidence import EvidenceComputation

# Set random seeds for reproducibility
seed = 123
torch.manual_seed(seed)
np.random.seed(seed)
pyro.set_rng_seed(seed)

## 1. Simulating the "Ground Truth" Data

To test our model comparison pipeline, we need observational data. We will simulate an outbreak using the **SEIR model** with known "true" parameters ($\beta=0.3$, $\gamma=0.15$, $\sigma=0.2$). 

To mimic real-world clinical or field data, we will add Gaussian noise to the peak infection rate returned by the simulator. This creates our set of 100 observations.

In [ ]:
# True parameters for the underlying SEIR dynamics
true_beta  = 0.3
true_gamma = 0.15
true_sigma = 0.2

noise_std = 0.05
n_obs = 100

# Initialize the SEIR simulator
seir_sim = SEIRSimulator(log_level="error")

# Generate the true output
params = torch.tensor([true_beta, true_gamma, true_sigma]).view(1, -1)
true_output = seir_sim.forward(params)

# Add noise to create synthetic observations
noise = torch.normal(0, noise_std, size=(n_obs,))
observed = true_output[0] + noise

observations = {"infection_rate": observed}

## 2. Training Emulators for Both Models

Bayesian calibration typically requires thousands of simulator evaluations. Because solving ODEs sequentially is computationally expensive, we will train a surrogate model (an emulator) for *both* the SIR and SEIR simulators.

We will use AutoEmulate to sample 1,500 input points for each model, run the full ODE simulators to get the corresponding outputs, and train a Radial Basis Function Gaussian Process (`GaussianProcessRBF`).

In [ ]:
n_train = 1500

# -------------------------------
# Train SEIR Emulator
# -------------------------------
print("Training SEIR Emulator...")
x_seir = seir_sim.sample_inputs(n_train)
y_seir, _ = seir_sim.forward_batch(x_seir)

ae_seir = AutoEmulate(
    x_seir,
    y_seir,
    models=[GaussianProcessRBF],
    device="cpu",
    log_level="error"
)
gp_seir = ae_seir.best_result().model

# -------------------------------
# Train SIR Emulator
# -------------------------------
print("Training SIR Emulator...")
sir_sim = Epidemic(log_level="error")

x_sir = sir_sim.sample_inputs(n_train)
y_sir, _ = sir_sim.forward_batch(x_sir)

ae_sir = AutoEmulate(
    x_sir,
    y_sir,
    models=[GaussianProcessRBF],
    device="cpu",
    log_level="error"
)
gp_sir = ae_sir.best_result().model

## 3. Bayesian Calibration (MCMC)

With our fast GP emulators trained, we can now perform Bayesian inference. 

We will use Markov Chain Monte Carlo (MCMC) to sample from the posterior parameter distributions of both models, conditioned strictly on the SEIR-generated observations we created in Step 1.

In [ ]:
warmup = 500
samples = 1500
chains = 2

# Calibrate the SEIR model
print("Calibrating SEIR model...")
bc_seir = BayesianCalibration(
    gp_seir,
    seir_sim.parameters_range,
    observations,
    observation_noise=noise_std**2
)

mcmc_seir = bc_seir.run_mcmc(
    warmup_steps=warmup,
    num_samples=samples,
    num_chains=chains
)

# Calibrate the SIR model
print("Calibrating SIR model...")
bc_sir = BayesianCalibration(
    gp_sir,
    sir_sim.parameters_range,
    observations,
    observation_noise=noise_std**2
)

mcmc_sir = bc_sir.run_mcmc(
    warmup_steps=warmup,
    num_samples=samples,
    num_chains=chains
)

## 4. Evidence Computation and the Bayes Factor

To formally compare the models, we need to compute the **Bayesian Evidence** (or Marginal Likelihood), denoted as $\mathcal{Z}$. The evidence quantifies the probability of observing the data given the model, integrating over the entire parameter space. It inherently penalizes overly complex models that do not offer a proportional increase in predictive power (Occam's razor).

We calculate this using the `EvidenceComputation` class, which implements a learned harmonic mean estimator via the `Harmonic` package (utilizing normalizing flows). 

Finally, we calculate the **Bayes Factor** $K$:
$$K = \exp(\ln \mathcal{Z}_{SEIR} - \ln \mathcal{Z}_{SIR})$$

A Bayes Factor $>1$ indicates evidence in favor of the SEIR model.

In [ ]:
training_prop = 0.5

# Compute evidence for SEIR
print("Computing evidence for SEIR...")
ec_seir = EvidenceComputation(
    mcmc_seir,
    bc_seir.model,
    training_proportion=training_prop
)
res_seir = ec_seir.run(epochs=100, verbose=False)

# Compute evidence for SIR
print("Computing evidence for SIR...")
ec_sir = EvidenceComputation(
    mcmc_sir,
    bc_sir.model,
    training_proportion=training_prop
)
res_sir = ec_sir.run(epochs=100, verbose=False)

# Calculate Bayes Factor
lnZ_seir = res_seir["ln_evidence"]
lnZ_sir  = res_sir["ln_evidence"]

bayes_factor = np.exp(lnZ_seir - lnZ_sir)

In [ ]:
print("\nSIR Model Posterior Summary:")
sir_summary = mcmc_sir.summary()
print(sir_summary)

print("\nSEIR Model Posterior Summary:")
seir_summary = mcmc_seir.summary()
print(seir_summary)

print("\n======================================")
print("MODEL SELECTION RESULTS")
print("======================================")
print(f"SEIR ln(Z): {lnZ_seir:.4f}")
print(f"SIR  ln(Z): {lnZ_sir:.4f}")
print(f"Bayes Factor (SEIR / SIR): {bayes_factor:.4e}")

## 5. Nested Sampling Evidence Check

To cross-check the Harmonic evidence estimates, we can compute again with a nested sampling implementation. This gives a benchmark for both the SEIR and SIR calibration problems.


In [ ]:
from dynesty import NestedSampler


def nested_sampling_evidence(bc, nlive=400):
    parameter_names = list(bc.parameter_range.keys())
    lower_bounds = np.array([bc.parameter_range[name][0] for name in parameter_names], dtype=float)
    upper_bounds = np.array([bc.parameter_range[name][1] for name in parameter_names], dtype=float)
    output_name = bc.output_names[0]
    observed_values = bc.observations[output_name].detach().cpu().numpy()
    observation_variance = float(bc.observation_noise[output_name])
    emulator = bc.emulator

    def prior_transform(unit_cube):
        return lower_bounds + unit_cube * (upper_bounds - lower_bounds)

    def loglike(theta):
        theta_tensor = torch.tensor(theta, dtype=torch.float32, device=bc.device).unsqueeze(0)
        mean = emulator.predict_mean(theta_tensor, with_grad=False)
        mean_value = float(mean.detach().cpu().reshape(-1)[0])
        residual = observed_values - mean_value
        return float(
            -0.5
            * np.sum(
                (residual**2) / observation_variance
                + np.log(2.0 * np.pi * observation_variance)
            )
        )

    sampler = NestedSampler(
        loglike,
        prior_transform,
        ndim=len(parameter_names),
        nlive=nlive,
        bound="multi",
        sample="rwalk",
    )
    sampler.run_nested(print_progress=False)

    results = sampler.results
    ln_evidence = float(results.logz[-1])
    ln_evidence_error = float(results.logzerr[-1])

    return {
        "ln_evidence": ln_evidence,
        "ln_evidence_error": ln_evidence_error,
    }


print("Computing nested sampling evidence for SEIR...")
ns_seir = nested_sampling_evidence(bc_seir)
print(
    f"SEIR nested ln(Z): {ns_seir['ln_evidence']:.4f} +/- {ns_seir['ln_evidence_error']:.4f}"
)

print("Computing nested sampling evidence for SIR...")
ns_sir = nested_sampling_evidence(bc_sir)
print(
    f"SIR  nested ln(Z): {ns_sir['ln_evidence']:.4f} +/- {ns_sir['ln_evidence_error']:.4f}"
)

nested_bayes_factor = np.exp(ns_seir["ln_evidence"] - ns_sir["ln_evidence"])
print("\n======================================")
print("NESTED SAMPLING RESULTS")
print("======================================")
print(f"SEIR nested ln(Z): {ns_seir['ln_evidence']:.4f}")
print(f"SIR  nested ln(Z): {ns_sir['ln_evidence']:.4f}")
print(f"Nested Bayes Factor (SEIR / SIR): {nested_bayes_factor:.4e}")
